# S18 — LLM Pretraining

**Week 10 · Mon Oct 26, 2026 · Module 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s18_llm_pretraining.ipynb)

Every cell below is a worked example from the [S18 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s18/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s18.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s18.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## Deduplication you can run


*Expected output starts with:* `doc 2 is an exact duplicate of doc 0 (after normalizing)`


In [ ]:
# Exact and near-duplicate detection on a toy corpus.
# Exact dedup: hash a normalized copy of each document.
# Near-dup detection: MinHash — estimate Jaccard similarity of shingle sets
# from k independent minimum hash values, without comparing sets directly.
import hashlib
import random

random.seed(0)

docs = [
    "The quick brown fox jumps over the lazy dog.",
    "Deep learning models improve with more data and compute.",
    "The quick brown fox jumps over the lazy dog.",          # exact dup of 0
    "  the QUICK brown fox jumps over the lazy dog. ",       # dup after normalizing
    "Deep learning models improve with more data & compute.",  # near-dup of 1
    "Neural networks are trained with gradient descent.",
    "A completely unrelated sentence about lasagna recipes.",
]

def normalize(text):
    return " ".join(text.lower().split())

# --- Stage 1: exact deduplication via content hashing ----------------------
seen, kept = {}, []
for i, doc in enumerate(docs):
    h = hashlib.sha1(normalize(doc).encode()).hexdigest()
    if h in seen:
        print(f"doc {i} is an exact duplicate of doc {seen[h]} (after normalizing)")
    else:
        seen[h] = i
        kept.append(i)
print(f"exact dedup: {len(docs)} docs -> {len(kept)} docs {kept}")

# --- Stage 2: MinHash near-duplicate estimation on the survivors -----------
def shingles(text, n=3):
    """Character n-grams of the normalized text, as a set."""
    t = normalize(text)
    return {t[i:i + n] for i in range(len(t) - n + 1)}

def jaccard(a, b):
    return len(a & b) / len(a | b)

K = 128  # number of hash functions = signature length
MERSENNE = (1 << 61) - 1
hash_params = [(random.randrange(1, MERSENNE), random.randrange(MERSENNE))
               for _ in range(K)]

def minhash_signature(shingle_set):
    """For each of K hash functions, keep the minimum hash over the set."""
    ints = [int.from_bytes(hashlib.sha1(s.encode()).digest()[:8], "big")
            for s in shingle_set]
    return [min((a * x + b) % MERSENNE for x in ints) for a, b in hash_params]

def minhash_estimate(sig_a, sig_b):
    """P[min-hash matches] = Jaccard similarity, so the match rate estimates it."""
    return sum(x == y for x, y in zip(sig_a, sig_b)) / K

sigs = {i: minhash_signature(shingles(docs[i])) for i in kept}
print(f"\npairwise similarity among the {len(kept)} surviving docs (threshold 0.6):")
print(f"{'pair':>8} {'true Jaccard':>13} {'MinHash est.':>13}  verdict")
for ai in range(len(kept)):
    for bi in range(ai + 1, len(kept)):
        i, j = kept[ai], kept[bi]
        true_j = jaccard(shingles(docs[i]), shingles(docs[j]))
        est_j = minhash_estimate(sigs[i], sigs[j])
        verdict = "NEAR-DUP" if est_j >= 0.6 else "-"
        if true_j > 0.05 or est_j >= 0.6:
            print(f"  ({i}, {j}) {true_j:>13.4f} {est_j:>13.4f}  {verdict}")

## Compute accounting: where 6ND comes from


*Expected output starts with:* `GPT-2 small-ish: d=768, layers=12, seq=1024`


In [ ]:
# Where does C ~= 6*N*D come from? Count the matrix-multiply FLOPs of one
# forward pass per token, layer by layer, and compare against 2*N (forward)
# and 6*N (forward + backward, since backward costs ~2x forward).
# A matmul that multiplies by an (m x n) weight costs ~2*m*n FLOPs per token.

def transformer_accounting(d, n_layer, vocab, seq):
    per_layer = {
        "attn QKV projections (3 * 2*d*d)": 6 * d * d,
        "attn output projection (2*d*d)":   2 * d * d,
        "attn scores QK^T (2*seq*d)":       2 * seq * d,
        "attn weighted sum AV (2*seq*d)":   2 * seq * d,
        "MLP up + down (2*d*4d + 2*4d*d)":  16 * d * d,
    }
    fwd = n_layer * sum(per_layer.values()) + 2 * d * vocab  # + LM head
    n_params = n_layer * 12 * d * d + vocab * d              # weights, tied embed
    return per_layer, fwd, n_params

configs = [("GPT-2 small-ish", 768, 12, 50257, 1024),
           ("7B-class",        4096, 32, 32000, 4096),
           ("7B-class, long ctx", 4096, 32, 32000, 131072)]

for name, d, n_layer, vocab, seq in configs:
    per_layer, fwd, n = transformer_accounting(d, n_layer, vocab, seq)
    print(f"{name}: d={d}, layers={n_layer}, seq={seq}")
    print(f"  params N ~ {n:.3e}   forward/token {fwd:.3e} FLOPs "
          f"(= {fwd / (2 * n):.2f} * 2N)")
    print(f"  fwd+bwd/token {3 * fwd:.3e} FLOPs (= {3 * fwd / (6 * n):.2f} * 6N)")

# The attention-score terms are the only ones that grow with seq, which is
# why 6*N*D is accurate at ordinary context lengths and drifts at long ones.

# --- From FLOPs to wall-clock: how long do real budgets take? --------------
PEAK = 312e12   # A100 bf16 peak, FLOP/s (NVIDIA datasheet)
MFU = 0.4       # optimistic sustained utilization for large-scale training

print(f"\n{'model':>22} {'N':>9} {'D':>9} {'tok/param':>10} {'C=6ND':>10} {'A100-days':>10}")
for name, N, D in [("Chinchilla (paper)", 70e9, 1.4e12),
                   ("LLaMA-7B (paper)", 7e9, 1.0e12),
                   ("Llama 2-7B (paper)", 7e9, 2.0e12)]:
    C = 6 * N * D
    days = C / (PEAK * MFU) / 86400
    print(f"{name:>22} {N:>9.1e} {D:>9.1e} {D / N:>10.1f} {C:>10.2e} {days:>10,.0f}")

## A toy scaling experiment you can run


*Expected output starts with:* `corpus: 127760 chars, vocab size 17`


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# --- Generate a corpus in a made-up language (no downloads) ---------------
g = torch.Generator().manual_seed(0)
syllables = ["ba", "ku", "ri", "ta", "no", "shi", "mo", "ler", "vin", "da"]
words = []
for i in range(50):
    n_syl = int(torch.randint(2, 4, (1,), generator=g))
    idx = torch.randint(0, len(syllables), (n_syl,), generator=g)
    words.append("".join(syllables[j] for j in idx))
corpus = " ".join(words[int(torch.randint(0, 50, (1,), generator=g))]
                  for _ in range(20000))

chars = sorted(set(corpus))
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in corpus])
n_train = int(0.9 * len(data))
train_data, val_data = data[:n_train], data[n_train:]
print(f"corpus: {len(corpus)} chars, vocab size {len(chars)}")

CTX = 32

def get_batch(split_data, batch_size, gen):
    ix = torch.randint(0, len(split_data) - CTX - 1, (batch_size,), generator=gen)
    x = torch.stack([split_data[i:i + CTX] for i in ix])
    y = torch.stack([split_data[i + 1:i + CTX + 1] for i in ix])
    return x, y

# --- A minimal decoder-only Transformer -----------------------------------
class Block(nn.Module):
    def __init__(self, d, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_head, batch_first=True)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        mask = nn.Transformer.generate_square_subsequent_mask(x.size(1))
        h = self.ln1(x)
        a, _ = self.attn(h, h, h, attn_mask=mask, need_weights=False)
        x = x + a
        return x + self.mlp(self.ln2(x))

class CharLM(nn.Module):
    def __init__(self, vocab, d, n_head, n_layer):
        super().__init__()
        self.tok = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(CTX, d)
        self.blocks = nn.Sequential(*[Block(d, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab)

    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.size(1)))
        return self.head(self.ln_f(self.blocks(x)))

def train_one(d, n_head, steps=300, batch_size=64):
    torch.manual_seed(0)
    gen = torch.Generator().manual_seed(1)
    model = CharLM(len(chars), d, n_head, n_layer=1)
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    for step in range(steps):
        x, y = get_batch(train_data, batch_size, gen)
        loss = F.cross_entropy(model(x).view(-1, len(chars)), y.view(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        eval_gen = torch.Generator().manual_seed(2)
        losses = []
        for _ in range(20):
            x, y = get_batch(val_data, batch_size, eval_gen)
            losses.append(F.cross_entropy(
                model(x).view(-1, len(chars)), y.view(-1)).item())
    return n_params, sum(losses) / len(losses)

print(f"{'d_model':>8} {'params':>10} {'val loss (nats/char)':>22}")
for d, n_head in [(8, 2), (32, 4), (128, 4)]:
    n_params, val_loss = train_one(d, n_head)
    print(f"{d:>8} {n_params:>10,} {val_loss:>22.4f}")
print(f"uniform-guess baseline: {math.log(len(chars)):.4f} nats/char")

## What a compute budget buys


*Expected output starts with:* `anchor: N = 70.0e9, D = 1.40e12  =>  C = 6*N*D = 5.88e+23 FLOPs`


In [ ]:
# Compute-optimal sizing, anchored to the Chinchilla paper's own choice.
# Hoffmann et al. (2022) report N_opt ~ C^0.50 and D_opt ~ C^0.50, and trained
# Chinchilla with N = 70e9 parameters on D = 1.4e12 tokens. Using the standard
# approximation C ~= 6*N*D FLOPs, we anchor the power laws at that point.

N_CHINCHILLA = 70e9      # parameters
D_CHINCHILLA = 1.4e12    # training tokens
C_CHINCHILLA = 6 * N_CHINCHILLA * D_CHINCHILLA

def compute_optimal(C, a=0.50, b=0.50):
    N = N_CHINCHILLA * (C / C_CHINCHILLA) ** a
    D = D_CHINCHILLA * (C / C_CHINCHILLA) ** b
    return N, D

print(f"anchor: N = 70.0e9, D = 1.40e12  =>  C = 6*N*D = {C_CHINCHILLA:.2e} FLOPs")
print(f"{'C (FLOPs)':>12} {'N_opt (params)':>16} {'D_opt (tokens)':>16} {'tokens/param':>13}")
for exp in range(19, 26):
    C = 10.0 ** exp
    N, D = compute_optimal(C)
    print(f"{C:>12.0e} {N:>16.3e} {D:>16.3e} {D / N:>13.1f}")

# What did Kaplan et al. (2020) recommend instead? Their fits favored growing
# N much faster than D (roughly N ~ C^0.73, D ~ C^0.27). Anchoring those
# exponents at the same point shows how differently the two papers allocate
# a 100x larger budget:
for a, b, label in [(0.73, 0.27, "Kaplan-style"), (0.50, 0.50, "Chinchilla-style")]:
    N, D = compute_optimal(100 * C_CHINCHILLA, a, b)
    print(f"{label:>17}: 100x compute -> N = {N:.2e}, D = {D:.2e} "
          f"({D / N:.1f} tokens/param)")

## Try it yourself

1. Add a fourth model size (`d = 512`) to the scaling script and a `steps=1200` run for each size. Does the loss gap between the two largest models grow or shrink with more steps? Time your runs.
2. Fit a line to `log(loss - L_floor)` versus `log(N)` for your runs, trying a few values of the irreducible-loss floor `L_floor`. How sensitive is the fitted exponent to that choice? (This sensitivity is a real issue in the published papers.)
3. Vary *data* instead of size: train the `d = 32` model on 1%, 10%, and 100% of the corpus (same steps). Which regime — data-limited or capacity-limited — does each run land in?
4. In the compute calculator, add an "inference-aware" column: fix `N` at half the compute-optimal value and solve for the `D` that uses the same total compute. How many tokens per parameter is that?


---

Full discussion of everything above: [S18 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s18/).
